# Delivery ETA: EDA and model training

This notebook explores the **synthetic, educational** dataset. It then runs the same cleaning and training code used by the app, so the notebook does not create a second inconsistent pipeline.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.train import clean_dataset, inspect_data, train_and_save

sns.set_theme(style='whitegrid', palette='deep')
raw_data = pd.read_csv(ROOT / 'data' / 'delivery_eta.csv')
raw_data.head()

: 

## Dataset overview, missingness, duplicates, and validation

Missing feature values are retained for the pipeline's imputers. Exact duplicates are removed, and impossible values or rows without a target are excluded. IQR outliers are flagged for review but retained because an unusually long trip may be a legitimate delivery.

In [ ]:
print(f'Shape: {raw_data.shape}')
display(raw_data.info())
display(raw_data.describe(include='all').T)
display(pd.Series(inspect_data(raw_data)['missing_values'], name='missing_values').sort_values(ascending=False))
print('Exact duplicate rows:', raw_data.duplicated().sum())
display(pd.Series(inspect_data(raw_data)['invalid_value_counts'], name='invalid_rows'))
display(pd.Series(inspect_data(raw_data)['iqr_outlier_flags_preserved'], name='IQR_flags_preserved'))
clean_data, cleaning_report = clean_dataset(raw_data)
print(cleaning_report)

## Target distribution and numerical relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.histplot(clean_data, x='delivery_time_min', kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Target: delivery time distribution')
sns.scatterplot(clean_data, x='distance_km', y='delivery_time_min', alpha=0.35, ax=axes[1])
axes[1].set_title('Delivery time vs distance')
sns.regplot(clean_data, x='preparation_time_min', y='delivery_time_min', scatter_kws={'alpha': 0.25}, line_kws={'color': 'crimson'}, ax=axes[2])
axes[2].set_title('Delivery time vs preparation time')
plt.tight_layout()

## Category comparisons

Boxplots make group-level patterns visible; they are descriptive only and do not establish causality.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 10))
plots = [('traffic_level', 'Traffic'), ('weather', 'Weather'), ('vehicle_type', 'Vehicle'), ('time_of_day', 'Time of day'), ('delivery_area', 'Delivery area'), ('day_of_week', 'Day of week')]
for axis, (column, title) in zip(axes.flat, plots):
    sns.boxplot(clean_data, x=column, y='delivery_time_min', ax=axis)
    axis.set_title(f'Delivery time vs {title.lower()}')
    axis.tick_params(axis='x', rotation=30)
plt.tight_layout()

## Numerical correlation analysis

Correlation is assessed only for numerical columns. Categorical variables are compared in the boxplots above.

In [ ]:
numerical_columns = ['distance_km', 'driver_experience_years', 'restaurant_rating', 'order_items', 'preparation_time_min', 'delivery_time_min']
plt.figure(figsize=(9, 7))
sns.heatmap(clean_data[numerical_columns].corr(), annot=True, cmap='vlag', center=0, fmt='.2f')
plt.title('Numerical feature correlations')
plt.show()

## Train, compare, and save the pipeline

The training module splits raw records first, engineers leakage-safe features, then fits imputers and encoders only on the training fold. MAE is the primary model-selection metric because it is directly interpretable as average minutes of error.

In [ ]:
comparison, metadata = train_and_save(ROOT / 'data' / 'delivery_eta.csv')
display(comparison.style.format({'MAE': '{:.3f}', 'RMSE': '{:.3f}', 'R2': '{:.3f}'}))
print('Best model:', metadata['best_model'])
print('Saved complete pipeline to:', ROOT / 'models' / 'eta_model.joblib')